# Guide 22: Hosting Capacity with OpenDSS

Hosting capacity is the maximum DER (solar, storage) that a transformer or feeder
can accommodate without violating voltage or thermal limits. This guide analyzes
hosting capacity results computed via iterative power flow.

**What you will learn:**
- How voltage-constrained hosting capacity differs from simplified thermal estimates
- How to interpret HC curves and identify limiting factors
- Distribution of HC across the service territory


## Setup

Run the cell below to clone the repository (Colab only).
If running locally, ensure you are in the `notebooks/` directory.


In [ ]:
# Colab setup: clone the repo and set working directory
import os
if not os.path.exists('Dynamic-Network-Model'):
    !git clone https://github.com/SGridworks/Dynamic-Network-Model.git
os.chdir('Dynamic-Network-Model/notebooks')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

RESULTS = '../sisyphean-power-and-light/network/results'


## Hosting Capacity Results


In [ ]:
hc = pd.read_parquet(f'{RESULTS}/hosting_capacity_powerflow.parquet')
print(f'Transformers analyzed: {len(hc)}')
print()
print(hc[['kva_rated', 'hc_kw', 'thermal_hc_kw', 'voltage_hc_kw', 'existing_pv_kw']].describe().round(1))


## HC Distribution

The histogram shows how hosting capacity varies across the service territory.
Smaller transformers (10-25 kVA residential) naturally have lower HC than
larger commercial units (250-500 kVA).


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(hc['hc_kw'], bins=40, color='#1C4855', edgecolor='white', alpha=0.8)
ax.axvline(hc['hc_kw'].median(), color='#5FCCDB', linestyle='--',
           label=f'Median: {hc["hc_kw"].median():.0f} kW')
ax.set_xlabel('Hosting Capacity (kW)')
ax.set_ylabel('Number of Transformers')
ax.set_title('Hosting Capacity Distribution')
ax.legend()
plt.tight_layout()
plt.show()


## HC vs Transformer Rating


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(hc['kva_rated'], hc['hc_kw'], alpha=0.5, c='#1C4855', s=20)
max_val = hc['kva_rated'].max() * 1.3
ax.plot([0, max_val], [0, max_val], 'r--', alpha=0.5, label='1:1 line (HC = rated kVA)')
ax.set_xlabel('Transformer Rating (kVA)')
ax.set_ylabel('Hosting Capacity (kW)')
ax.set_title('Hosting Capacity vs Transformer Rating')
ax.legend()
plt.tight_layout()
plt.show()


## Hosting Capacity Curves

Each curve shows how voltage changes as PV injection increases at a single transformer.
The hosting capacity is the PV level where the curve crosses 1.05 pu.


In [ ]:
curves = pd.read_parquet(f'{RESULTS}/hosting_capacity_curves.parquet')

examples = hc.nsmallest(1, 'kva_rated')['transformer_id'].tolist() + \
           hc[hc['kva_rated'].between(70, 80)].head(1)['transformer_id'].tolist() + \
           hc.nlargest(1, 'kva_rated')['transformer_id'].tolist()

fig, ax = plt.subplots(figsize=(10, 5))
for xid in examples:
    c = curves[curves['transformer_id'] == xid]
    kva = hc[hc['transformer_id'] == xid]['kva_rated'].iloc[0]
    ax.plot(c['pv_kw'], c['voltage_pu'], marker='o', markersize=4,
            label=f'{xid} ({kva:.0f} kVA)')

ax.axhline(1.05, color='#E74C3C', linestyle='--', alpha=0.7, label='Voltage limit (1.05 pu)')
ax.set_xlabel('PV Injection (kW)')
ax.set_ylabel('Secondary Bus Voltage (pu)')
ax.set_title('Hosting Capacity Curves')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## Simplified vs Power Flow Comparison

The simplified thermal method estimates HC as 80% of transformer kVA rating.
Power flow analysis accounts for voltage constraints and network impedance,
often yielding different results.


In [ ]:
hc['simplified_hc'] = hc['kva_rated'] * 0.8

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(hc['simplified_hc'], hc['hc_kw'], alpha=0.5, c='#1C4855', s=20)
max_val = hc['simplified_hc'].max() * 1.2
ax.plot([0, max_val], [0, max_val], 'r--', alpha=0.5, label='1:1 (agreement)')
ax.set_xlabel('Simplified HC (kVA * 0.8) [kW]')
ax.set_ylabel('Power Flow HC [kW]')
ax.set_title('Simplified vs Power-Flow Hosting Capacity')
ax.legend()
plt.tight_layout()
plt.show()

ratio = hc['hc_kw'] / hc['simplified_hc']
print(f'Mean ratio (PF / simplified): {ratio.mean():.2f}')
print(f'Cases where PF < simplified: {(ratio < 1.0).sum()} ({(ratio < 1.0).mean()*100:.0f}%)')


## Key Takeaways

1. **Voltage constraints matter**: Power flow HC can differ significantly from simplified thermal estimates.
2. **HC scales with transformer size**: Larger transformers naturally host more DER.
3. **Network location matters**: Transformers at feeder endpoints may have lower HC due to voltage sensitivity.
4. **Existing DER**: 264 transformers already have solar PV, reducing remaining headroom.

See Guide 21 for power flow fundamentals and Guide 23 for loss analysis.
